# 08 — T và TH trên AML-Small-HI (3 seed 42–44, 2×T4)

- **T:** FraudGT gốc + temporal neighbor sampling strict-past.
- **TH:** T + 8 historical features strict-past của H.

Notebook kiểm tra temporal leakage trước, sau đó chạy T trên GPU 0 và TH trên GPU 1; mỗi mô hình chạy độc lập 3 seed 42–44.

In [ ]:
SEED = 42
REPEATS = 3  # seed 42, 43, 44
NUM_THREADS = 2
NUM_WORKERS = 2
print(f'Seeds: {SEED}..{SEED + REPEATS - 1} | repeats: {REPEATS}')

## 1. Môi trường và dependency

In [ ]:
import platform, sys, subprocess, torch
print('Python:', sys.version)
print('Platform:', platform.platform())
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {p.name}; VRAM={p.total_memory / 1024**3:.2f} GiB')
subprocess.run(['nvidia-smi'], check=False)

torch_version = torch.__version__.split('+')[0]
cuda_tag = 'cu' + torch.version.cuda.replace('.', '') if torch.version.cuda else 'cpu'
wheel_url = f'https://data.pyg.org/whl/torch-{torch_version}+{cuda_tag}.html'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pyg_lib', 'torch_scatter', 'torch_sparse', '-f', wheel_url], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch_geometric', 'torchmetrics', 'yacs', 'datatable',
                'pandas', 'matplotlib', 'wandb', 'ogb', 'tensorboardX',
                'pyyaml'], check=True)
print('Dependencies installed.')

## 2. Lấy mã nguồn và gắn dữ liệu

In [ ]:
from pathlib import Path
from shutil import copy2
import os

REPO_URL = 'https://github.com/mhiunguyen/TH-FraudGT.git'
repo = Path('/kaggle/working/TH-FraudGT')
if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
os.chdir(repo)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Commit:', commit)

required = [
    repo / 'configs/AML-Small-HI/AML-Small-HI-Temporal-T4.yaml',
    repo / 'configs/AML-Small-HI/AML-Small-HI-Temporal-History-T4.yaml',
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise RuntimeError('Repository chưa có mã T/TH mới: ' + str(missing))

candidates = list(Path('/kaggle/input').rglob('HI-Small_Trans.csv'))
if not candidates:
    raise FileNotFoundError('Hãy Add Input bộ IBM AML; thiếu HI-Small_Trans.csv.')
destination = repo / 'data/AML/HI-Small_Trans.csv'
destination.parent.mkdir(parents=True, exist_ok=True)
if not destination.exists() or destination.stat().st_size != candidates[0].stat().st_size:
    copy2(candidates[0], destination)
print('Dataset:', destination, '| MiB:', round(destination.stat().st_size / 1024**2, 1))

## 3. Audit temporal sampler trên đồ thị nhỏ
Cell phải in `PASS`: cạnh cùng thời điểm và cạnh tương lai không được xuất hiện trong neighborhood.

In [ ]:
from torch_geometric.data import HeteroData
from torch_geometric.loader import LinkNeighborLoader

toy = HeteroData()
toy['node'].x = torch.ones(5, 1)
toy['node'].num_nodes = 5
edge_index = torch.tensor([[0, 1, 2, 0, 3], [1, 2, 3, 3, 4]])
edge_time = torch.tensor([1., 2., 5., 5., 6.])
for edge_type, index in [(('node','to','node'), edge_index),
                         (('node','rev_to','node'), edge_index.flip(0))]:
    toy[edge_type].edge_index = index
    toy[edge_type].edge_attr = torch.ones(edge_index.shape[1], 1)
    toy[edge_type].timestamps = edge_time
target_time = torch.tensor([5.])
strict_cutoff = torch.nextafter(target_time, torch.full_like(target_time, -torch.inf))
audit_loader = LinkNeighborLoader(
    toy, num_neighbors=[-1, -1],
    edge_label_index=(('node','to','node'), torch.tensor([[0], [3]])),
    edge_label=torch.tensor([0]), edge_label_time=strict_cutoff,
    time_attr='timestamps', temporal_strategy='last', disjoint=True,
    batch_size=1, shuffle=False)
audit_batch = next(iter(audit_loader))
sampled_times = []
for edge_type in audit_batch.edge_types:
    sampled_times.extend(audit_batch[edge_type].timestamps.tolist())
assert sampled_times and max(sampled_times) < target_time.item(), sampled_times
print('PASS strict-past audit | sampled timestamps:', sorted(set(sampled_times)))

## 4. Sinh config thực tế và tạo cache tuần tự

In [ ]:
import copy, gc, time, yaml
cfg_root = repo / 'configs/AML-Small-HI'
t_cfg = yaml.safe_load((cfg_root / 'AML-Small-HI-Temporal-T4.yaml').read_text(encoding='utf-8'))
th_cfg = yaml.safe_load((cfg_root / 'AML-Small-HI-Temporal-History-T4.yaml').read_text(encoding='utf-8'))
for cfg in [t_cfg, th_cfg]:
    cfg['out_dir'] = str(repo / 'results')
    cfg['dataset']['dir'] = str(repo / 'data')
    cfg['seed'] = SEED
    cfg['num_threads'] = NUM_THREADS
    cfg['num_workers'] = NUM_WORKERS
    cfg['wandb']['use'] = False

# Fairness guard: T và TH chỉ được khác add_history.
t_compare = copy.deepcopy(t_cfg); th_compare = copy.deepcopy(th_cfg)
t_compare['dataset']['add_history'] = th_compare['dataset']['add_history']
assert t_compare == th_compare, 'T/TH còn khác thông số ngoài add_history.'
assert t_cfg['train']['sampler'] == 'temporal_link_neighbor'
assert t_cfg['train']['temporal_strict'] is True

generated = Path('/kaggle/working/generated_configs')
generated.mkdir(parents=True, exist_ok=True)
T_CFG = generated / 'AML-Small-HI-T-seeds42-44.yaml'
TH_CFG = generated / 'AML-Small-HI-TH-seeds42-44.yaml'
T_CFG.write_text(yaml.safe_dump(t_cfg, sort_keys=False), encoding='utf-8')
TH_CFG.write_text(yaml.safe_dump(th_cfg, sort_keys=False), encoding='utf-8')
print('T:', T_CFG, '| history:', t_cfg['dataset']['add_history'])
print('TH:', TH_CFG, '| history:', th_cfg['dataset']['add_history'])

sys.path.insert(0, str(repo))
from fraudGT.datasets.aml_dataset import AMLDataset
for name, add_history in [('T', False), ('TH', True)]:
    started = time.time()
    cache = AMLDataset(root=str(repo / 'data/AML'), name='Small-HI',
                       reverse_mp=True, add_ports=True, add_history=add_history)
    for split in ['train', 'val', 'test']:
        assert hasattr(cache[split]['node','rev_to','node'], 'timestamps')
    print(f'{name} cache ready in {(time.time()-started)/60:.1f} min')
    del cache; gc.collect()

## 5. Train T và TH song song

In [ ]:
jobs = [
    {'name':'T', 'cfg':T_CFG, 'gpu':0, 'tag':'T-Seeds42-44', 'log':Path('/kaggle/working/T_seeds42_44.log')},
    {'name':'TH', 'cfg':TH_CFG, 'gpu':1 if torch.cuda.device_count() >= 2 else 0,
     'tag':'TH-Seeds42-44', 'log':Path('/kaggle/working/TH_seeds42_44.log')},
]
def command(job):
    return [sys.executable, '-u', '-m', 'fraudGT.main', '--cfg', str(job['cfg']),
            '--repeat', str(REPEATS), '--gpu', str(job['gpu']), 'name_tag', job['tag']]
def run_jobs(selected):
    handles = []
    for job in selected:
        stream = job['log'].open('w', encoding='utf-8')
        process = subprocess.Popen(command(job), cwd=repo, stdout=stream,
                                   stderr=subprocess.STDOUT, text=True)
        handles.append((job, process, stream))
        print(f"Started {job['name']} on GPU {job['gpu']} | PID {process.pid}")
    started = time.time()
    while any(p.poll() is None for _, p, _ in handles):
        time.sleep(60)
        print(f'[heartbeat] {(time.time()-started)/60:.0f} min',
              [(j['name'], 'running' if p.poll() is None else 'done') for j,p,_ in handles], flush=True)
        subprocess.run(['nvidia-smi','--query-gpu=index,memory.used,utilization.gpu',
                        '--format=csv,noheader'], check=False)
    failed = []
    for job, process, stream in handles:
        stream.close()
        if process.returncode != 0:
            failed.append(job['name'])
            print('\n'.join(job['log'].read_text(errors='replace').splitlines()[-80:]))
    if failed: raise RuntimeError('Failed: ' + ', '.join(failed))
if torch.cuda.device_count() >= 2:
    run_jobs(jobs)
else:
    for job in jobs: run_jobs([job])
for job in jobs:
    log_text = job['log'].read_text(encoding='utf-8', errors='replace')
    for seed in range(SEED, SEED + REPEATS):
        assert f'seed={seed}' in log_text, f"{job['name']} thiếu seed {seed}"
print('Training completed and all seeds 42–44 are present in both logs.')

## 6. Tổng hợp kết quả fixed threshold 0.50 và validation-selected

In [ ]:
import pandas as pd
summarizer = repo / 'scripts/summarize_thresholds.py'
frames, evidence = [], [T_CFG, TH_CFG]
for job in jobs:
    run_dir = repo / 'results' / f"{job['cfg'].stem}-{job['tag']}-gpu{job['gpu']}"
    for protocol, extra in [('fixed_0.50', ['--fixed-threshold','0.50']),
                            ('validation_selected', [])]:
        output = Path(f"/kaggle/working/summary_{job['name']}_{protocol}_seeds42_44.csv")
        subprocess.run([sys.executable, str(summarizer), str(run_dir),
                        '--output', str(output)] + extra, check=True)
        frame = pd.read_csv(output)
        frame.insert(0, 'model', job['name']); frame.insert(1, 'protocol', protocol)
        frames.append(frame); evidence.append(output)
    evidence.append(job['log'])
results = pd.concat(frames, ignore_index=True)
RESULTS = Path('/kaggle/working/summary_T_TH_seeds42_44.csv')
results.to_csv(RESULTS, index=False); evidence.append(RESULTS)
display(results[['model','protocol','seed','best_epoch','threshold','val_f1',
                 'test_f1','test_precision','test_recall','test_auc']])
print('Summary:', RESULTS)

## 7. Đóng gói bằng chứng

In [ ]:
import shutil
bundle = Path('/kaggle/working/T_TH_3Seeds_T4x2_artifacts')
bundle.mkdir(exist_ok=True)
for path in evidence:
    if Path(path).exists(): shutil.copy2(path, bundle / Path(path).name)
archive = shutil.make_archive(str(bundle), 'zip', bundle)
print('Download:', archive)